## SVOD_TEMPLATE_CREATION

###📍IMPORTANT: Always check the order of WBTV and Foundry data

In [ ]:
# ============================================================
# 📚 LIBRARIES - EXTERNAL
# ============================================================
import os
import time
import warnings
import logging

import pandas as pd
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter

warnings.filterwarnings("ignore")
os.environ["TRANSFORMERS_NO_TF"] = "1"

# ============================================================
# LOGGING
# ============================================================
"""
Creates and configures a module-level logger for the SVOD template creation
workflow.

The logger records informational messages and writes them to a log file named
SVOD.log. The log file captures timestamp, logger name, log level, and message
details, which helps with debugging, monitoring, and troubleshooting pipeline
execution.

The handler check prevents duplicate file handlers from being added when the
module is imported or executed multiple times.

Configuration
-------------
Logger Name
    Uses __name__, which sets the logger name based on the current module.

Log Level
    logging.INFO is used to capture informational messages and above, including
    warnings, errors, and critical messages.

Log File
    SVOD.log is created in write mode. Existing log content is overwritten each
    time the script starts.

Log Format
    Each log entry follows this format:
    timestamp - logger name - log level - message

Side Effects
------------
Creates or overwrites the SVOD.log file.
Adds a FileHandler to the logger if no handlers already exist.
"""

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

if not logger.handlers:
    formatter = logging.Formatter(
        "%(asctime)s - %(name)s - %(levelname)s - %(message)s"
    )
    file_handler = logging.FileHandler("SVOD.log", mode="w")
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)

# ============================================================
# 📚 LIBRARIES - OWN FUNCTIONS
# ============================================================
"""Importing the matching_pipeline class from the Packages module to handle the SVOD template creation workflow."""
from Packages import matching_pipeline

pipeline = matching_pipeline()

# ============================================================
# INPUTS - DYNAMIC SYNOPSIS LIMITS
# ============================================================
"""
Collects dynamic synopsis column requirements and runs the SVOD template pipeline.

This block prompts the user to enter the number of synopsis columns needed and
the character limit for each synopsis column. It validates that all entered
values are numeric and greater than zero.

After collecting the synopsis configuration, the block asks the user to select
the SVOD template type, runs the template pipeline, selects an output folder,
saves the generated DataFrame as a CSV file, and prompts the user to confirm
whether file verification is complete.

Variables
---------
synopsis_count : int
    Number of synopsis columns required in the output template.

synopsis_limits : list
    List of character limits entered by the user for each synopsis column.

template_type : str
    Selected SVOD template type. Possible values are "series" or "movie".

final_df : pandas.DataFrame
    Final DataFrame returned by the SVOD template pipeline.

output_folder : str
    Folder path selected by the user for saving output files.

format_check_path : str
    Full file path where the generated format check CSV file is saved.

english_path : str
    Path reference assigned to the generated format check CSV file.

temp_path : str
    Path reference assigned to the selected output folder.

verify : str
    User confirmation input for file verification status.

Side Effects
------------
Prompts the user for input values.
Runs the SVOD template pipeline.
Opens an output folder selection dialog.
Saves the final DataFrame as a CSV file.
Writes informational messages to the logger.
Prints validation messages for invalid user input.

"""

logger.info("Enter dynamic synopsis column details")

while True:
    try:
        synopsis_count = int(input("Enter number of synopsis columns needed: ").strip())

        if synopsis_count <= 0:
            print("❌ Number of synopsis columns must be greater than 0.")
            continue

        break

    except ValueError:
        print("❌ Invalid input. Please enter a numeric value.")

synopsis_limits = []

for i in range(1, synopsis_count + 1):
    while True:
        try:
            limit = int(input(f"Enter character limit for synopsis column {i}: ").strip())

            if limit <= 0:
                print("❌ Character limit must be greater than 0.")
                continue

            synopsis_limits.append(limit)
            break

        except ValueError:
            print("❌ Invalid input. Please enter numeric character limit.")

template_type = pipeline.ask_content_type()

final_df = pipeline.run_template_pipeline(
    content_type=template_type,
    top_k=5,
    ce_threshold=0.75
)

output_folder = pipeline.select_output_folder()
format_check_path = os.path.join(output_folder, "WBTVD or WB2B or FOUNDRY - Format.csv")
english_path = format_check_path
temp_path = output_folder

final_df.to_csv(format_check_path, index=False)

logger.info("File verification Done")
verify = input("File verification Done [y/n]: ").strip().lower()

# ============================================================
# HELPER FUNCTIONS
# ============================================================
def get_base_columns():
    """
    Returns the base column structure for the SVOD output template.

    This function defines the standard set of columns that should appear at the
    beginning of the generated SVOD template. These columns represent common
    metadata fields such as serial number, category, season, episode, title,
    localized title, internal reference, and release date.

    Returns
    -------
    list
        A list of base column names used in the SVOD template output.
    """

    return [
        "Sr.No.",
        "Category",
        "Season",
        "Episode",
        "Source Title (Long Description)",
        "Localized Title",
        "WM Internal Reference",
        "US Release Date"
    ]


def apply_dynamic_synopsis_columns(df, source_df, synopsis_limits, is_translation=False):
    """
    Dynamically creates synopsis source and translation columns.

    For each entered limit, creates:
    1. Synopsis ({limit}) SOURCE ({limit} Character Limit)
    2. Source char. Counter ({limit})
    3. Synopsis ({limit}) TRANSLATION ({limit} Character Limit)
    4. Translation char. Count ({limit})

    Same structure is applied for English and translation sheets.
    """

    base_columns = get_base_columns()

    for col in base_columns:
        if col not in df.columns:
            df[col] = ""

    output_df = df[base_columns].copy()
    column_order = base_columns.copy()
    row_count = len(output_df)

    for limit in synopsis_limits:
        source_col = f"Synopsis ({limit}) SOURCE ({limit} Character Limit)"
        source_count_col = f"Source char. Counter ({limit})"

        translation_col = f"Synopsis ({limit}) TRANSLATION ({limit} Character Limit)"
        translation_count_col = f"Translation char. Count ({limit})"

        # Add columns in final order first
        column_order.extend([
            source_col,
            source_count_col,
            translation_col,
            translation_count_col
        ])

        # SOURCE synopsis column
        output_df[source_col] = pipeline.char_limit(limit, source_df, "yes")

        # TRANSLATION synopsis column
        if is_translation:
            output_df[translation_col] = pipeline.char_limit(limit, df)
        else:
            output_df[translation_col] = ""

        # Dynamic Excel column letters
        source_excel_col = get_column_letter(column_order.index(source_col) + 1)
        translation_excel_col = get_column_letter(column_order.index(translation_col) + 1)

        # LEN formulas
        output_df[source_count_col] = [
            f"=LEN({source_excel_col}{row_num})"
            for row_num in range(2, row_count + 2)
        ]

        output_df[translation_count_col] = [
            f"=LEN({translation_excel_col}{row_num})"
            for row_num in range(2, row_count + 2)
        ]

    return output_df[column_order]


def prepare_english_dataframe(df, template_type, synopsis_limits):
    """
    Prepares the English source DataFrame for SVOD template output creation.

    This function standardizes Season and Episode columns, formats release dates,
    sorts the data based on the selected template type, creates base output
    columns, maps title and reference fields, and applies dynamic synopsis
    columns based on the provided character limits.

    For series templates, the data is sorted by Season and Episode. Season-level
    rows are formatted by appending the season number to the source title. For
    movie templates, the data is sorted by the available output title column.

    Parameters
    ----------
    df : pandas.DataFrame
        The input DataFrame generated from the SVOD template pipeline.

    template_type : str
        The selected template type. Expected values are "series" or "movie".

    synopsis_limits : list
        A list of character limits used to create dynamic synopsis columns in
        the output template.

    Returns
    -------
    pandas.DataFrame
        The prepared English output DataFrame with base template columns and
        dynamic synopsis columns applied.

    Raises
    ------
    ValueError
        Raised when no valid title column is found for output creation.
    """

    df = df.copy()

    if "Season" not in df.columns:
        df["Season"] = 0

    if "Episode" not in df.columns:
        df["Episode"] = 0

    df["Season"] = pd.to_numeric(df["Season"], errors="coerce").fillna(0).astype(int)
    df["Episode"] = pd.to_numeric(df["Episode"], errors="coerce").fillna(0).astype(int)

    if "Primary Release Date" in df.columns:
        df["Primary Release Date"] = pd.to_datetime(
            df["Primary Release Date"],
            errors="coerce"
        )

    if template_type == "series":
        df = df.sort_values(["Season", "Episode"]).reset_index(drop=True)
    else:
        title_col = pipeline.get_output_title_column(df)

        if title_col:
            df = df.sort_values(by=[title_col]).reset_index(drop=True)
        else:
            df = df.reset_index(drop=True)

    df["Sr.No."] = range(1, len(df) + 1)
    df["Category"] = "WB"

    title_col = pipeline.get_output_title_column(df)

    if title_col is None:
        raise ValueError("❌ No valid title column found for output creation.")

    df["Source Title (Long Description)"] = df[title_col]

    if template_type == "series":
        df["Source Title (Long Description)"] = [
            f"{title}: Season {season}" if episode == 0 and season != 0 else title
            for title, episode, season in zip(
                df["Source Title (Long Description)"],
                df["Episode"],
                df["Season"]
            )
        ]

    df["Localized Title"] = ""

    if "MPM Number" in df.columns:
        df["WM Internal Reference"] = df["MPM Number"]
    elif "uuid" in df.columns:
        df["WM Internal Reference"] = df["uuid"]
    else:
        df["WM Internal Reference"] = ""

    if "Primary Release Date" in df.columns:
        df["US Release Date"] = df["Primary Release Date"].dt.strftime("%Y-%m-%d")
    else:
        df["US Release Date"] = ""

    return apply_dynamic_synopsis_columns(
        df=df,
        source_df=df,
        synopsis_limits=synopsis_limits,
        is_translation=False
    )


def prepare_translation_dataframe(
    df_trans,
    english_df,
    template_type,
    synopsis_limits,
    lang_name
):
    """
    Prepares a translated language DataFrame for SVOD template output creation.

    This function aligns translated Foundry data with the English source
    DataFrame using the uuid field. It standardizes Season and Episode values,
    merges English metadata with translated content, creates the required base
    output columns, maps localized titles, formats release dates, and applies
    dynamic synopsis columns based on the provided character limits.

    For series templates, season-level rows are formatted by appending the season
    number to the source title. The translated title is placed in the Localized
    Title column when available.

    Parameters
    ----------
    df_trans : pandas.DataFrame
        The translated Foundry DataFrame for a specific language.

    english_df : pandas.DataFrame
        The prepared English source DataFrame used as the base reference for
        matching translated rows.

    template_type : str
        The selected template type. Expected values are "series" or "movie".

    synopsis_limits : list
        A list of character limits used to create dynamic synopsis columns in
        the output template.

    lang_name : str
        The language name associated with the translated DataFrame. This is used
        in error messages when title or merge validation fails.

    Returns
    -------
    pandas.DataFrame
        The prepared translated output DataFrame with base template columns,
        localized title values, release date, internal reference, and dynamic
        synopsis columns applied.

    Raises
    ------
    ValueError
        Raised when English and translation data do not align correctly for the
        provided language.

    ValueError
        Raised when no valid title column is found for the translated sheet.
    """


    df_trans = df_trans.copy()
    english_df = english_df.copy()

    # ========================================================
    # PREPARE TRANSLATION SEASON / EPISODE
    # ========================================================
    if "season-number" in df_trans.columns:
        df_trans["Season"] = pd.to_numeric(
            df_trans["season-number"],
            errors="coerce"
        ).fillna(0).astype(int)
    else:
        df_trans["Season"] = 0

    if "episode-number" in df_trans.columns:
        df_trans["Episode"] = pd.to_numeric(
            df_trans["episode-number"],
            errors="coerce"
        ).fillna(0).astype(int)
    else:
        df_trans["Episode"] = 0

    df_trans = df_trans.sort_values(["Season", "Episode"]).reset_index(drop=True)

    # ========================================================
    # PREPARE ENGLISH SEASON / EPISODE
    # ========================================================
    if "Season" not in english_df.columns:
        english_df["Season"] = 0

    if "Episode" not in english_df.columns:
        english_df["Episode"] = 0

    english_df["Season"] = pd.to_numeric(
        english_df["Season"],
        errors="coerce"
    ).fillna(0).astype(int)

    english_df["Episode"] = pd.to_numeric(
        english_df["Episode"],
        errors="coerce"
    ).fillna(0).astype(int)

    if "Primary Release Date" in english_df.columns:
        english_df["Primary Release Date"] = pd.to_datetime(
            english_df["Primary Release Date"],
            errors="coerce"
        )
    else:
        english_df["Primary Release Date"] = ""

    # ========================================================
    # MERGE ENGLISH WITH TRANSLATION DATA
    # ========================================================
    required_english_cols = ["uuid", "Season", "Episode", "Primary Release Date"]
    optional_english_cols = []

    if "*Title name" in english_df.columns:
        optional_english_cols.append("*Title name")

    if "MPM Number" in english_df.columns:
        optional_english_cols.append("MPM Number")

    merge_cols = required_english_cols + optional_english_cols

    df = pd.merge(
        english_df[merge_cols],
        df_trans,
        on="uuid",
        how="left",
        suffixes=("_eng", "_trans")
    )

    if "*Title name" in df.columns and df["*Title name"].isna().any():
        raise ValueError(f"❌ English mismatch in {lang_name}")

    df["Sr.No."] = range(1, len(df) + 1)
    df["Category"] = "WB"

    if "Season_eng" in df.columns:
        df["Season"] = df["Season_eng"]

    if "Episode_eng" in df.columns:
        df["Episode"] = df["Episode_eng"]

    title_col = pipeline.get_output_title_column(df)

    if title_col is None:
        if "*Title name" in df.columns:
            title_col = "*Title name"
        else:
            raise ValueError(f"❌ No title column found in translated sheet: {lang_name}")

    df["Source Title (Long Description)"] = df[title_col]

    if "main-title" in df.columns:
        df["Localized Title"] = df["main-title"].fillna("")
    else:
        df["Localized Title"] = ""

    if template_type == "series":
        df["Source Title (Long Description)"] = [
            f"{title}: Season {int(season)}" if episode == 0 and season != 0 else title
            for title, episode, season in zip(
                df["Source Title (Long Description)"],
                df["Episode"],
                df["Season"]
            )
        ]

    if "MPM Number" in df.columns:
        df["WM Internal Reference"] = df["MPM Number"]
    elif "uuid" in df.columns:
        df["WM Internal Reference"] = df["uuid"]
    else:
        df["WM Internal Reference"] = ""

    if "Primary Release Date" in df.columns:
        df["US Release Date"] = pd.to_datetime(
            df["Primary Release Date"],
            errors="coerce"
        ).dt.strftime("%Y-%m-%d")
    else:
        df["US Release Date"] = ""

    return apply_dynamic_synopsis_columns(
        df=df,
        source_df=english_df,
        synopsis_limits=synopsis_limits,
        is_translation=True
    )


def create_dynamic_summary(output_file, synopsis_limits):
    """
    Creates a dynamic summary DataFrame for all sheets in the generated SVOD
    output workbook.

    This function reads every sheet from the provided Excel output file, except
    the Info sheet, and creates a summary row for each sheet. For each synopsis
    character limit provided by the user, it checks whether the corresponding
    source and translation synopsis columns are available, validates synopsis
    length, checks translation availability, and calculates word count for source
    text where translation is missing.

    The function does not use hardcoded short, long, or too-long synopsis logic.
    Instead, it dynamically builds column names and validation checks based on
    the character limits entered by the user.

    Parameters
    ----------
    output_file : str
        The full path of the Excel output file containing SVOD template sheets.

    synopsis_limits : list
        A list of character limits entered by the user. Each limit is used to
        dynamically identify source, translation, and validation columns.

    Returns
    -------
    pandas.DataFrame
        A summary DataFrame containing availability and validation status for
        each sheet and each configured synopsis character limit. The summary also
        includes a total Word_Count value for source synopsis text where the
        corresponding translation is missing.
    """


    summary_rows = []
    sheets = pd.read_excel(output_file, sheet_name=None)

    for sheet_name, df in sheets.items():
        if sheet_name == "Info":
            continue

        summary_data = {
            "Sheet": sheet_name
        }

        total_word_count = 0

        for limit in synopsis_limits:
            source_col = f"Synopsis ({limit}) SOURCE ({limit} Character Limit)"
            translation_col = f"Synopsis ({limit}) TRANSLATION ({limit} Character Limit)"

            source_status_col = f"Synopsis ({limit}) ENGLISH ({limit} Character Limit)"
            translation_status_col = f"Synopsis ({limit}) TRANSLATION ({limit} Character Limit)"
            translation_len_status_col = f"Synopsis ({limit}) TRANS_len ({limit} Character Limit)"

            if source_col in df.columns:
                source_series = df[source_col]
                source_status = pipeline.synopsis_status(source_series, limit)
            else:
                source_series = pd.Series(dtype="object")
                source_status = "Not available"

            if translation_col in df.columns:
                translation_series = df[translation_col]
                translation_status = pipeline.availability_status(translation_series)
                translation_len_status = pipeline.synopsis_status(translation_series, limit)
            else:
                translation_series = pd.Series(dtype="object")
                translation_status = "Not available"
                translation_len_status = "Not available"

            if source_col in df.columns and translation_col in df.columns:
                word_count = pipeline.conditional_word_count(
                    source_series,
                    translation_series
                )
            else:
                word_count = 0

            total_word_count += word_count

            summary_data[translation_status_col] = translation_status
            summary_data[source_status_col] = source_status
            summary_data[translation_len_status_col] = translation_len_status

        summary_data["Word_Count"] = total_word_count
        summary_rows.append(summary_data)

    return pd.DataFrame(summary_rows)


def format_sheet_safely(ws):
    """
    Safely apply formatting to a worksheet.

    format_sheet_openpyxl() now detects synopsis columns
    dynamically from the headers, so no synopsis limits
    need to be passed.
    """
    try:
        pipeline.format_sheet_openpyxl(ws)

    except Exception as e:
        logger.warning(
            f"Formatting skipped for sheet '{ws.title}': {e}"
        )


# ============================================================
# MAIN TEMPLATE CREATION
# ============================================================

"""
Creates the final SVOD Excel template after the format verification step.

This block runs only when the user confirms that file verification is complete.
It optionally processes multilanguage Foundry files, creates the final Excel
output file, writes the English template sheet, writes translation sheets when
available, and handles errors for open or locked Excel files.

The generated workbook is saved using the series name entered by the user.

Variables
---------
verify : str
    User confirmation value. The block executes only when this value is "y".

only_english : str
    User input indicating whether a multilanguage folder is available.
    Expected values are "y" or "n".

path : str or None
    Folder path containing multilanguage CSV files. This is populated only when
    the user confirms that the multilanguage folder is available.

series_name : str
    Series name entered by the user.

series : str
    Output file prefix created by adding "SVOD_" before the series name.

en_path : str
    Path of the English source CSV file generated during format verification.

temp : str
    Output folder path where the final Excel template will be saved.

output_file : str
    Full file path of the generated SVOD Excel template.

english : pandas.DataFrame
    English source data loaded from the verified CSV file.

english_prepared : pandas.DataFrame
    English data transformed into the final template structure.

written_sheets : list
    List of sheet names successfully written to the Excel workbook.

Side Effects
------------
Prompts the user for multilanguage availability and series name.
Opens a folder selection dialog if multilanguage files are available.
Deletes an existing output Excel file with the same name.
Reads the verified English CSV file.
Creates a new Excel workbook.
Writes the English sheet to the workbook.
Writes translation sheets for valid multilanguage CSV files.
Writes an Info sheet if no valid translation data is found.
Logs warnings and errors for skipped or failed translation files.
Raises an error if the output Excel file is open or locked.
"""

if verify == "y":
    only_english = input("Is the multilanguage folder available? [y/n]: ").strip().lower()
    path = None

    if only_english == "y":
        path = pipeline.pick_multilang_folder()

    series_name = input("Enter the Series name: ").strip()
    series = f"SVOD_{series_name}"

    en_path = english_path
    temp = temp_path

    output_file = os.path.join(temp, f"{series}_Template.xlsx")

    # =========================
    # DELETE OLD FILE
    # =========================
    if os.path.exists(output_file):
        try:
            os.remove(output_file)
        except PermissionError:
            raise RuntimeError("❌ Excel file is open. Close it and retry.")

    # =========================
    # LOAD ENGLISH MASTER
    # =========================
    english = pd.read_csv(en_path)

    english_prepared = prepare_english_dataframe(
        df=english,
        template_type=template_type,
        synopsis_limits=synopsis_limits
    )

    written_sheets = []

    try:
        with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

            # ====================================
            # ENGLISH SHEET
            # ====================================
            english_prepared.to_excel(writer, sheet_name="English", index=False)
            written_sheets.append("English")

            # ====================================
            # TRANSLATION FILES
            # ====================================
            if only_english == "y":

                for file in os.listdir(path):
                    if not file.endswith(".csv"):
                        continue

                    try:
                        parts = file.split("-")

                        if len(parts) < 3 or "export_" not in parts[2]:
                            logger.warning(f"Skipping file with unexpected format: {file}")
                            continue

                        name = (
                            parts[2]
                            .split("export_")[1]
                            .replace("_", " ")
                            .replace(".csv", "")
                            .strip()
                        )

                        if name == "English United States":
                            continue

                        df_trans = pd.read_csv(os.path.join(path, file))

                        translated_df = prepare_translation_dataframe(
                            df_trans=df_trans,
                            english_df=english,
                            template_type=template_type,
                            synopsis_limits=synopsis_limits,
                            lang_name=name
                        )

                        safe_sheet_name = name[:31]

                        translated_df.to_excel(
                            writer,
                            sheet_name=safe_sheet_name,
                            index=False
                        )

                        written_sheets.append(safe_sheet_name)

                    except Exception as e:
                        logger.error(f"Failed processing {file}: {e}")

                if len(written_sheets) == 1:
                    pd.DataFrame({
                        "Message": ["No valid translation data found"]
                    }).to_excel(
                        writer,
                        sheet_name="Info",
                        index=False
                    )

    except PermissionError:
        raise RuntimeError("❌ Excel file is open. Close it before running.")

    time.sleep(2)

    # ============================================================
    # SUMMARY SHEET - DYNAMIC
    # ============================================================

    """
    Creates and writes the dynamic Info summary sheet into the final SVOD Excel template.

    This block generates a summary DataFrame using the configured synopsis character
    limits and appends it to the existing Excel workbook as an Info sheet. If an Info
    sheet already exists, it is replaced with the newly generated summary.

    Variables
    ---------
    summary_df : pandas.DataFrame
        Summary DataFrame created from the output workbook. It contains synopsis
        availability, synopsis validation status, translation availability, and word
        count details for each sheet.

    output_file : str
        Full file path of the final SVOD Excel workbook.

    synopsis_limits : list
        List of character limits used to dynamically evaluate synopsis columns.

    Side Effects
    ------------
    Reads the generated Excel workbook.
    Creates a dynamic summary based on available sheets and synopsis columns.
    Appends or replaces the Info sheet in the output workbook.
    """

    summary_df = create_dynamic_summary(
        output_file=output_file,
        synopsis_limits=synopsis_limits
    )

    with pd.ExcelWriter(
        output_file,
        mode="a",
        engine="openpyxl",
        if_sheet_exists="replace"
    ) as writer:
        summary_df.to_excel(writer, sheet_name="Info", index=False)

    # ============================================================
    # FORMAT SHEETS
    # ============================================================
    """
    Formats the generated SVOD Excel workbook and saves the final output file.

    This block opens the completed Excel workbook, applies worksheet formatting to
    each successfully written template sheet, saves the updated workbook, logs a
    success message, and prints the final output file location.

    If file verification was not confirmed, the template creation process is stopped
    and a message is displayed to the user.

    Variables
    ---------
    wb : openpyxl.workbook.workbook.Workbook
        Workbook object loaded from the generated SVOD Excel output file.

    output_file : str
        Full file path of the generated SVOD Excel workbook.

    written_sheets : list
        List of worksheet names that were successfully written to the workbook.

    ws : openpyxl.worksheet.worksheet.Worksheet
        Worksheet object selected from the workbook for formatting.

    series : str
        Name used for the generated SVOD template file.

    Side Effects
    ------------
    Opens the generated Excel workbook.
    Applies formatting to each written worksheet.
    Saves the updated workbook.
    Writes a success message to the log file.
    Prints the final output location to the console.
    Stops template creation if file verification is not confirmed.
    """


    wb = load_workbook(output_file)

    for name in written_sheets:
        if name in wb.sheetnames:
            ws = wb[name]
            format_sheet_safely(ws)

    wb.save(output_file)

    logger.info(f"Template '{series}' created successfully.")
    print(f"✅ Template '{series}' created successfully at:\n{output_file}")

else:
    print("❌ File verification was not confirmed. Template creation stopped.")

Select folder to save output files...
✅ Folder cleaned successfully
Selected Output Folder: C:/Users/mshanmugam/OneDrive - Warner Bros. Discovery/JUPYTER_PY/AI Projects/Automated Metadata Template Creation (SVOD)/Output_folder
Formatting applied on sheet 'English'. Validation columns found: 4. Highlighted cells: 16
✅ Template 'SVOD_Nikita_(S4)_pt_PT' created successfully at:
C:/Users/mshanmugam/OneDrive - Warner Bros. Discovery/JUPYTER_PY/AI Projects/Automated Metadata Template Creation (SVOD)/Output_folder\SVOD_Nikita_(S4)_pt_PT_Template.xlsx
